# Metrics

A metric is a function `(example, prediction) -> float` that scores how good the model's output is.  
Every optimizer in DSPy needs one — this is what 'better' means.

In [1]:
import numpy
import dspy
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

lm = dspy.LM('ollama_chat/llama3.1:8b', api_base='http://localhost:11434')
dspy.configure(lm=lm)

## Step 1 — Load dataset and build examples

We filter Yahoo Answers to 3 unambiguous topics and map them to our domain labels.

In [3]:
from datasets import load_dataset

TOPIC_MAP = {
    1: "science",   # Science & Mathematics
    5: "sports",    # Sports
    9: "politics",  # Politics & Government
}

raw = load_dataset("yahoo_answers_topics", split="test")
filtered = [r for r in raw if r["topic"] in TOPIC_MAP]

examples = [
    dspy.Example(
        question=r["question_title"],
        domain=TOPIC_MAP[r["topic"]]
    ).with_inputs("question")
    for r in filtered
]

print(f"Total examples: {len(examples)}")
print(f"Sample: {examples[0]}")

Total examples: 18000
Sample: Example({'question': 'Why does Zebras have stripes?', 'domain': 'science'}) (input_keys={'question'})


## Step 2 — Train / dev split

In [4]:
import random
random.seed(42)
random.shuffle(examples)

trainset = examples[:50]
devset   = examples[50:100]

print(f"Train: {len(trainset)} | Dev: {len(devset)}")

Train: 50 | Dev: 50


## Step 3 — Define the module

Updated to use politics instead of history to match the dataset.

In [5]:
class DomainClassifier(dspy.Signature):
    """Classify the domain of the question."""

    question: str = dspy.InputField(desc="A user's question")
    domain: Literal["science", "sports", "politics"] = dspy.OutputField(desc="The domain the question belongs to")


class Classifier(dspy.Module):
    def __init__(self):
        self.classify = dspy.Predict(DomainClassifier)

    def forward(self, question):
        return self.classify(question=question)

## Step 4 — Write the metric

The metric receives an `example` (ground truth) and a `prediction` (model output).  
It returns a float — here 1.0 for correct, 0.0 for wrong.

In [7]:
def domain_accuracy(example, prediction, trace=None):
    return float(example.domain == prediction.domain)

## Step 5 — Evaluate over the devset

In [8]:
from dspy.evaluate import Evaluate

classifier = Classifier()

evaluate = Evaluate(
    devset=devset,
    metric=domain_accuracy,
    num_threads=4,
    display_progress=True,
)

score = evaluate(classifier)
print(f"Accuracy: {score.score:.1f}%")

Average Metric: 39.00 / 50 (78.0%): 100%|██████████| 50/50 [01:15<00:00,  1.51s/it]

2026/09/03 13:37:57 INFO dspy.evaluate.evaluate: Average Metric: 39.0 / 50 (78.0%)



Accuracy: 78.0%
